# Synthetic dataset training run

# Continual Learning â€” Colab Training Notebook

**Before running:**
1. Set runtime to **GPU** (Runtime â†’ Change runtime type â†’ T4 GPU or better)
2. Edit the **Configuration** cell below
3. Run all cells top-to-bottom

Checkpoints and the dataset cache are stored on **Google Drive** and survive session restarts.
If a checkpoint already exists for your `RUN_ID`, training will automatically resume from it.

## Configuration
Edit the values below before running the notebook.

In [1]:
# ============================================================
#  CONFIGURE EVERYTHING HERE
# ============================================================

# Training method â€” choose one:
#   "full_ft"  â€” fine-tune all parameters (catastrophic forgetting baseline)
#   "lora"     â€” parameter-efficient LoRA adapters
#   "smf"      â€” frozen backbone + sparse trainable memory
#   "casm"     â€” versioned memory bank with contradiction-aware routing
METHOD = "casm"

# HuggingFace model (must have approved access)
MODEL_NAME = "meta-llama/Llama-3.2-1B"

# Unique experiment name â€” checkpoints are stored under this ID
RUN_ID = "llama1b_synthetic_casm_similarity_08"

# Periods to train â€” full sequence is all four in order
# Each period is one temporal snapshot of the synthetic dataset
PERIODS = ["2018", "2020"]  # full run: ["2018", "2020", "2022", "2024"]

# Google Drive folder â€” checkpoints and dataset cache go here
DRIVE_DIR = "/content/drive/MyDrive/continual_learning/synthetic"

# ---- Dataset fraction ----
# Fraction of the dataset to use for both training and evaluation.
# The fraction is applied proportionally and independently to the changed and
# unchanged probe splits, so training and evaluation always cover the same
# examples.  None means use all data; 0.1 means use 10% of each split.
DATASET_FRACTION = .3  # synthetic dataset is small (605 facts) â€” use all data

# ---- Core hyperparameters ----
BATCH_SIZE              = 1
GRAD_ACCUM_STEPS        = 4       # effective batch size = 4
LEARNING_RATE           = 5e-4
EPOCHS_PER_PERIOD       = 15     # passes over all passages
MAX_PASSAGES_PER_PERIOD = None    # use all available passages
PRECISION               = "bfloat16"
SEED                    = 42

# ---- Activation capture ----
# Records per-layer hidden-state activations after each period.
# Output saved to periods/<period>/activations.pt and activation_metadata.json
# alongside eval outputs â€” copy to another machine for plotting, no GPU needed.
CAPTURE_ACTIVATIONS = True

# ---- LoRA settings (only used when METHOD == "lora") ----
LORA_R              = 16
LORA_ALPHA          = 32
LORA_DROPOUT        = 0.05
LORA_TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]

# ---- SMF settings (only used when METHOD == "smf") ----
# Llama-3.2-1B has 16 transformer layers (indices 0-15)
SMF_MEMORY_SIZE            = 64
SMF_SPARSITY_RATIO         = 0.1
SMF_UPDATE_LAYERS          = [4, 8, 12]  # mid-to-late layers
SMF_REGULARIZATION_WEIGHT  = 0.01
SMF_LEARNING_RATE          = 1e-3

# ---- CASM settings (only used when METHOD == "casm") ----
CASM_NUM_SLOTS              = 8
CASM_ROUTER_HIDDEN_SIZE     = 256
CASM_TOP_K                  = 2
CASM_ROUTER_TEMPERATURE     = 1.0
CASM_SPARSITY_WEIGHT        = 0.01
CASM_OVERLAP_WEIGHT         = 0.01
CASM_BRANCH_ON_CONTRADICTION = True
CASM_MEMORY_SIZE            = 64
CASM_ROUTER_TYPE            = 'similarity'  # 'mlp' = CASMRouter (learned, default), 'similarity' = SimilarityRouter (cosine, no training)

## Setup

In [2]:
# Mount Google Drive for persistent checkpoint and dataset storage
from google.colab import drive
drive.mount("/content/drive")

import os

CHECKPOINT_DIR   = os.path.join(DRIVE_DIR, "checkpoints")
DATASET_CACHE_DIR = os.path.join(DRIVE_DIR, "dataset_cache")
REPO_DIR         = "/content/Continual-Learning"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(DATASET_CACHE_DIR, exist_ok=True)

print(f"Checkpoint dir:  {CHECKPOINT_DIR}")
print(f"Dataset cache:   {DATASET_CACHE_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoint dir:  /content/drive/MyDrive/continual_learning/synthetic/checkpoints
Dataset cache:   /content/drive/MyDrive/continual_learning/synthetic/dataset_cache


In [3]:
# Clone the repo (or pull if already cloned this session)
import subprocess

REPO_URL = "https://github.com/athirai-s/Continual-Learning"

if os.path.exists(os.path.join(REPO_DIR, ".git")):
    print("Repo already cloned â€” pulling latest changes...")
    result = subprocess.run(
        ["git", "-C", REPO_DIR, "pull"],
        capture_output=True, text=True
    )
    print(result.stdout.strip() or result.stderr.strip())
else:
    print(f"Cloning {REPO_URL} ...")
    result = subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        capture_output=True, text=True
    )
    print(result.stdout.strip() or result.stderr.strip())
    if result.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{result.stderr}")

print("Repo ready.")

Repo already cloned â€” pulling latest changes...
Updating 4dd2669..faeb53a
Fast-forward
 .github/workflows/ci.yml |  5 +----
 training/casm_model.py   | 21 +++++++++++++++------
 2 files changed, 16 insertions(+), 10 deletions(-)
Repo ready.


In [4]:
# Install dependencies not pre-installed on Colab
# transformers 5.x is required â€” Colab ships with 4.x by default
!pip install -q "transformers>=5.3.0" "peft>=0.18.1" "trl>=0.29.0" "datasets>=4.6.1"

# Add the repo to the Python path so imports work
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Dependencies installed. Repo on path.")

Dependencies installed. Repo on path.


## Dataset

Training passages come from `data/augmented/synthetic/<period>.csv` â€” these are
committed directly in the repo, so no download is needed.

Evaluation probes come from `data/probes.json`, which is also committed in the repo.
`SyntheticDataset` loads this file directly â€” no zip extraction needed.
The synthetic dataset contains 605 facts with ~150 changed probes per period boundary.

In [5]:
import os

# Verify augmented training CSVs (data/augmented/synthetic/<period>.csv)
aug_dir = os.path.join(REPO_DIR, "data", "augmented", "synthetic")
for period in PERIODS:
    csv_path = os.path.join(aug_dir, f"{period}.csv")
    assert os.path.exists(csv_path), f"Synthetic CSV missing: {csv_path}"
    print(f"  {period}.csv: OK ({os.path.getsize(csv_path):,} bytes)")

# Verify probes file
probes_json = os.path.join(REPO_DIR, "data", "probes.json")
assert os.path.exists(probes_json), (
    f"probes.json not found at {probes_json}. "
    "Run data/build_probes.py to generate it."
)
print(f"probes.json: OK ({os.path.getsize(probes_json):,} bytes)")
print("All synthetic dataset files present.")


  2018.csv: OK (73,841 bytes)
  2020.csv: OK (74,448 bytes)
probes.json: OK (1,243,132 bytes)
All synthetic dataset files present.


## HuggingFace Authentication

Required for gated models like Llama. Log in with a token that has **read** access to
`meta-llama/Llama-3.2-1B`. You can create a token at https://huggingface.co/settings/tokens

In [6]:
from huggingface_hub import login
login()  # will prompt for your HuggingFace access token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## Training

In [7]:
import os

aug_dir = os.path.join(REPO_DIR, "data", "augmented", "synthetic")
all_ok = True
for period in PERIODS:
    path = os.path.join(aug_dir, f"{period}.csv")
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0
    status = "OK" if exists else "MISSING"
    print(f"  {period}.csv: {status} ({size:,} bytes)")
    if not exists:
        all_ok = False

if all_ok:
    print("All synthetic CSVs found â€” ready to train.")
else:
    raise FileNotFoundError(
        "One or more synthetic CSVs are missing. "
        "Check that data/augmented/synthetic/<period>.csv files are committed to the repo."
    )


  2018.csv: OK (73,841 bytes)
  2020.csv: OK (74,448 bytes)
All synthetic CSVs found â€” ready to train.


In [8]:
# Build TrainConfig from configuration variables set at the top
from training.train_config import TrainConfig

config_kwargs = dict(
    run_id=RUN_ID,
    model_name=MODEL_NAME,
    method=METHOD,
    dataset_name="synthetic",
    precision=PRECISION,
    batch_size=BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    epochs_per_period=EPOCHS_PER_PERIOD,
    max_passages_per_period=MAX_PASSAGES_PER_PERIOD,
    dataset_fraction=DATASET_FRACTION,
    log_every_n_steps=10,
    eval_after_each_period=True,
    capture_activations=CAPTURE_ACTIVATIONS,
    seed=SEED,
)

if METHOD == "lora":
    config_kwargs.update(
        lora_r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        lora_target_modules=LORA_TARGET_MODULES,
    )
elif METHOD == "smf":
    config_kwargs.update(
        smf_memory_size=SMF_MEMORY_SIZE,
        smf_sparsity_ratio=SMF_SPARSITY_RATIO,
        smf_update_layers=SMF_UPDATE_LAYERS,
        smf_regularization_weight=SMF_REGULARIZATION_WEIGHT,
        smf_learning_rate=SMF_LEARNING_RATE,
        smf_freeze_backbone=True,
    )
elif METHOD == "casm":
    config_kwargs.update(
        casm_num_slots=CASM_NUM_SLOTS,
        casm_router_hidden_size=CASM_ROUTER_HIDDEN_SIZE,
        casm_top_k=CASM_TOP_K,
        casm_router_temperature=CASM_ROUTER_TEMPERATURE,
        casm_sparsity_weight=CASM_SPARSITY_WEIGHT,
        casm_overlap_weight=CASM_OVERLAP_WEIGHT,
        casm_branch_on_contradiction=CASM_BRANCH_ON_CONTRADICTION,
        casm_memory_size=CASM_MEMORY_SIZE,
        casm_router_type=CASM_ROUTER_TYPE,
    )

cfg = TrainConfig(**config_kwargs)
cfg.validate()

print(f"Method:               {cfg.method}")
print(f"Model:                {cfg.model_name}")
print(f"Periods:              {PERIODS}")
print(f"Batch size:           {cfg.batch_size}")
print(f"Grad accum steps:     {cfg.grad_accum_steps}  (eff. batch = {cfg.batch_size * cfg.grad_accum_steps})")
print(f"Learning rate:        {cfg.learning_rate}")
print(f"Epochs per period:    {cfg.epochs_per_period}")
print(f"Max passages:         {cfg.max_passages_per_period}")
print(f"Dataset fraction:     {cfg.dataset_fraction if cfg.dataset_fraction is not None else 'all (None)'}")
print(f"Capture activations:  {cfg.capture_activations}")
print(f"Checkpoint dir:       {CHECKPOINT_DIR}")

Method:               casm
Model:                meta-llama/Llama-3.2-1B
Periods:              ['2018', '2020']
Batch size:           1
Grad accum steps:     4  (eff. batch = 4)
Learning rate:        0.0005
Epochs per period:    15
Max passages:         None
Dataset fraction:     0.3
Capture activations:  True
Checkpoint dir:       /content/drive/MyDrive/continual_learning/synthetic/checkpoints


In [9]:
# Detect existing checkpoint for automatic resume
import json

RESUME_FROM = None
run_root = os.path.join(CHECKPOINT_DIR, RUN_ID)
latest_json = os.path.join(run_root, "latest.json")

if os.path.exists(latest_json):
    with open(latest_json) as f:
        pointer = json.load(f)
    last_period = pointer.get("last_period", "unknown")
    print(f"Existing checkpoint found (last period: {last_period}).")
    print(f"Training will resume from: {run_root}")
    RESUME_FROM = run_root
else:
    print("No existing checkpoint â€” starting fresh.")

No existing checkpoint â€” starting fresh.


In [10]:
# ── Data verification ─────────────────────────────────────────────────────
# Loads each period using the same factory as training — augmented CSVs,
# same dataset_fraction — so counts here match what training will actually see.
from training.train_runner import build_synthetic_dataset

print(f"Method: {cfg.method}")
print(f"Dataset: {cfg.dataset_name}")
print(f"Periods: {PERIODS}")
print(f"Dataset fraction: {cfg.dataset_fraction if cfg.dataset_fraction is not None else 'all (None)'}")
print()

for period in PERIODS:
    ds = build_synthetic_dataset(period, cfg)
    ds.load("changed")
    ds.load("unchanged")

    passages   = ds.get_train_passages()
    changed    = ds.get_probes("changed")
    unchanged  = ds.get_probes("unchanged")

    print(f"Period: {period}")
    print(f"  Training passages : {len(passages)}")
    print(f"  Changed probes    : {len(changed)}")
    print(f"  Unchanged probes  : {len(unchanged)}")

    if passages:
        print(f"  Sample passage    : {passages[0][:120]!r}")
    if changed:
        p = changed[0]
        print(f"  Sample changed    : [{p.subject}] {p.relation} -> {p.current_value!r}  (was {p.previous_value!r})")
    if unchanged:
        p = unchanged[0]
        print(f"  Sample unchanged  : [{p.subject}] {p.relation} -> {p.current_value!r}")
    print()

print("Data looks correct — proceed to training.")


Method: casm
Dataset: synthetic
Periods: ['2018', '2020']
Dataset fraction: 0.3

  dataset_fraction=0.300: 182/605 changed probes, 0/0 unchanged probes, 182/605 training passages
Period: 2018
  Training passages : 182
  Changed probes    : 182
  Unchanged probes  : 0
  Sample passage    : 'The primary berth depth of Azure Tide Harbor is 12 fathoms. Ships can dock safely there. It accommodates large vessels.'
  Sample changed    : [Azure Tide Harbor] primary berth depth -> '12 fathoms'  (was None)

  dataset_fraction=0.300: 45/150 changed probes, 137/455 unchanged probes, 182/605 training passages
Period: 2020
  Training passages : 182
  Changed probes    : 45
  Unchanged probes  : 137
  Sample passage    : 'The primary berth depth of Azure Tide Harbor is 12 fathoms. Ships can dock safely. It is a busy port.'
  Sample changed    : [Misty Fjord Logistics] dock capacity -> '25 ships'  (was None)
  Sample unchanged  : [Azure Tide Harbor] primary berth depth -> '12 fathoms'

Data looks corr

In [11]:
from training.train_runner import (
    run_training,
    build_real_model_and_tokenizer,
    load_real_model_and_tokenizer,
    build_synthetic_dataset,
)

dataset_factory = build_synthetic_dataset
print("Dataset factory: synthetic CSVs (data/augmented/synthetic/)")
results = run_training(
    cfg,
    model_factory=build_real_model_and_tokenizer,
    resume_model_factory=load_real_model_and_tokenizer,
    dataset_factory=dataset_factory,
    checkpoint_dir=CHECKPOINT_DIR,
    training_units=PERIODS,
    resume_from=RESUME_FROM,
)

print("\n" + "="*50)
print("TRAINING SUMMARY")
print("="*50)
for i, r in enumerate(results):
    period = PERIODS[i] if i < len(PERIODS) else f"period_{i}"
    print(f"\n{period}:")
    loss = r.get("train_loss_final")
    print(f"  Final loss:        {loss:.4f}" if loss is not None else "  Final loss:        N/A")
    print(f"  Passages trained:  {r.get('n_passages_trained', 'N/A')}")
    print(f"  Train time (s):    {r.get('train_duration_sec', 0):.1f}")
    if "evaluation" in r:
        for split, eval_result in r["evaluation"].items():
            if isinstance(eval_result, dict):
                plasticity = eval_result.get("plasticity")
                stability  = eval_result.get("stability")
                token_f1   = eval_result.get("token_f1")
            else:
                plasticity = eval_result.plasticity
                stability  = eval_result.stability
                token_f1   = eval_result.token_f1
            p = f"{plasticity:.3f}" if plasticity is not None else "N/A"
            s = f"{stability:.3f}"  if stability  is not None else "N/A"
            f = f"{token_f1:.3f}"   if token_f1   is not None else "N/A"
            print(f"  Eval [{split:9s}]: plasticity={p}  stability={s}  token_f1={f}")
    print(f"  Checkpoint:        {r.get('checkpoint_path', 'N/A')}")

Dataset factory: synthetic CSVs (data/augmented/synthetic/)
Training config:
TrainConfig(model_name='meta-llama/Llama-3.2-1B', method='casm', dataset_name='synthetic', precision='bfloat16', learning_rate=0.0005, batch_size=1, grad_accum_steps=4, epochs_per_period=15, grad_clip=1.0, warmup_steps=100, min_passage_length=100, deduplicate=True, weighted_sampling=False, max_passages_per_period=None, dataset_fraction=0.3, shuffle_by_relation=True, contradiction_batch_frac=0.25, run_id='llama1b_synthetic_casm_similarity_08', log_every_n_steps=10, eval_after_each_period=True, capture_activations=True, checkpoint_dir='/content/drive/MyDrive/continual_learning/synthetic/checkpoints', checkpoint_every_n_optimizer_steps=None, seed=42, lora_r=None, lora_alpha=None, lora_dropout=0.05, lora_target_modules=None, smf_memory_size=None, smf_sparsity_ratio=None, smf_update_layers=None, smf_regularization_weight=0.01, smf_freeze_backbone=True, smf_learning_rate=None, casm_num_slots=8, casm_router_hidden_si

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Saved config to /content/drive/MyDrive/continual_learning/synthetic/checkpoints/llama1b_synthetic_casm_similarity_08_config.json


=== Training unit: 2018 ===
  dataset_fraction=0.300: 182/605 changed probes, 0/0 unchanged probes, 182/605 training passages
  Passages:      2220 (15 epoch(s))
  Optimizer steps: 555  |  eff_batch=4  |  lr=0.00e+00
  Device:          cuda
  step  10/555 (  1.8%)  loss=4.2141  lr=5.00e-05   1068 tok/s  elapsed=00:01  ETA=01:37
  step  20/555 (  3.6%)  loss=4.3351  lr=1.00e-04   1009 tok/s  elapsed=00:02  ETA=01:16
  step  30/555 (  5.4%)  loss=4.4134  lr=1.50e-04    985 tok/s  elapsed=00:03  ETA=01:09
  step  40/555 (  7.2%)  loss=4.1648  lr=2.00e-04   1133 tok/s  elapsed=00:05  ETA=01:04
  step  50/555 (  9.0%)  loss=4.4216  lr=2.50e-04   1044 tok/s  elapsed=00:06  ETA=01:01
  step  60/555 ( 10.8%)  loss=3.6832  lr=3.00e-04   1102 tok/s  elapsed=00:07  ETA=00:58
  step  70/555 ( 12.6%)  loss=4.0306  lr=3.50e-04   1047 tok/s  elapsed=00:08  ETA=00:56
  ste

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  dataset_fraction=0.300: 182/605 changed probes, 0/0 unchanged probes, 182/605 training passages
Activations saved: 10 probes × 16 layers → /content/drive/MyDrive/continual_learning/synthetic/checkpoints/llama1b_synthetic_casm_similarity_08/periods/2018/activations.pt
Training result:
  Final loss: 3.3130974769592285
  Passages trained: 2220
  Contradiction passages: 0
  Train duration (sec): 59.63
Checkpoint saved to: /content/drive/MyDrive/continual_learning/synthetic/checkpoints/llama1b_synthetic_casm_similarity_08/checkpoints/ckpt-000001

=== Training unit: 2020 ===
  dataset_fraction=0.300: 45/150 changed probes, 137/455 unchanged probes, 182/605 training passages
  Passages:      2310 (15 epoch(s))
  Optimizer steps: 578  |  eff_batch=4  |  lr=0.00e+00
  Device:          cuda
  step  10/578 (  1.7%)  loss=5.1858  lr=5.00e-05    679 tok/s  elapsed=00:01  ETA=01:44
  step  20/578 (  3.5%)  loss=5.4617  lr=1.00e-04    673 tok/s  elapsed=00:03  ETA=01:38
  step  30/578 (  5.2%)  los

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  dataset_fraction=0.300: 45/150 changed probes, 137/455 unchanged probes, 182/605 training passages
Activations saved: 20 probes × 16 layers → /content/drive/MyDrive/continual_learning/synthetic/checkpoints/llama1b_synthetic_casm_similarity_08/periods/2020/activations.pt
Training result:
  Final loss: 4.235795974731445
  Passages trained: 2310
  Contradiction passages: 42
  Train duration (sec): 90.34
Checkpoint saved to: /content/drive/MyDrive/continual_learning/synthetic/checkpoints/llama1b_synthetic_casm_similarity_08/checkpoints/ckpt-000002

Done.

TRAINING SUMMARY

2018:
  Final loss:        3.3131
  Passages trained:  2220
  Train time (s):    59.6
  Eval [changed  ]: plasticity=0.000  stability=0.005  token_f1=0.004
  Eval [unchanged]: plasticity=0.000  stability=0.000  token_f1=0.000
  Checkpoint:        /content/drive/MyDrive/continual_learning/synthetic/checkpoints/llama1b_synthetic_casm_similarity_08/checkpoints/ckpt-000001

2020:
  Final loss:        4.2358
  Passages trai

In [12]:
# Print eval summary from the completed run (no retraining needed)
print("\n" + "="*50)
print("EVAL SUMMARY")
print("="*50)
for i, r in enumerate(results):
    period = PERIODS[i] if i < len(PERIODS) else f"period_{i}"
    print(f"\n{period}:")
    if "evaluation" in r:
        for split, eval_result in r["evaluation"].items():
            if isinstance(eval_result, dict):
                plasticity = eval_result.get("plasticity")
                stability  = eval_result.get("stability")
                token_f1   = eval_result.get("token_f1")
            else:
                plasticity = eval_result.plasticity
                stability  = eval_result.stability
                token_f1   = eval_result.token_f1
            p = f"{plasticity:.3f}" if plasticity is not None else "N/A"
            s = f"{stability:.3f}"  if stability  is not None else "N/A"
            f = f"{token_f1:.3f}"   if token_f1   is not None else "N/A"
            print(f"  [{split:9s}]: plasticity={p}  stability={s}  token_f1={f}")
    else:
        print("  (no evaluation results)")


EVAL SUMMARY

2018:
  [changed  ]: plasticity=0.000  stability=0.005  token_f1=0.004
  [unchanged]: plasticity=0.000  stability=0.000  token_f1=0.000

2020:
  [changed  ]: plasticity=0.000  stability=0.000  token_f1=0.000
  [unchanged]: plasticity=0.000  stability=0.000  token_f1=0.000


In [13]:
import torch
from training.train_runner import load_real_model_and_tokenizer

checkpoint_path = results[0]["checkpoint_path"]
model, tokenizer = load_real_model_and_tokenizer(cfg, checkpoint_path)
model.eval()

device = next(model.parameters()).device

sample_dataset = dataset_factory(PERIODS[-1], cfg)
sample_dataset.load("changed")
probes = sample_dataset.get_probes("changed")[:5]

for probe in probes:
    inputs = tokenizer(probe.prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
        )
    generated = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    print(f"Prompt:   {probe.prompt}")
    print(f"Expected: {probe.ground_truth}")
    print(f"Got:      {generated!r}")
    print()

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


  dataset_fraction=0.300: 45/150 changed probes, 137/455 unchanged probes, 182/605 training passages


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

Prompt:   Who is the light intensity of Starfish Reef Marker?
Expected: 6000 lumens
Got:      'The light intensity of Star'



Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Prompt:   Who is the annual sediment removal of Deepwater Dredging Co.?
Expected: 18000 cubic meters
Got:      'The annual sediment removal of'



Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Prompt:   Who is the fog signal duration of Oracle Rocks Beacon?
Expected: 12 seconds
Got:      'What is the fog signal'

Prompt:   Who is the rotation period of Seafire Lighthouse?
Expected: 25 seconds
Got:      'The Seafire L'



In [14]:
from collections import defaultdict

def param_summary(model):
    total_params = 0
    trainable_params = 0
    groups = defaultdict(lambda: {"total": 0, "trainable": 0})

    for name, param in model.named_parameters():
        n = param.numel()
        top = name.split(".")[0]
        total_params += n
        groups[top]["total"] += n
        if param.requires_grad:
            trainable_params += n
            groups[top]["trainable"] += n

    col = 42
    print(f"\n{'Module':<{col}} {'Total params':>14} {'Trainable':>12} {'%':>7}")
    print("-" * (col + 36))
    for group, counts in sorted(groups.items()):
        t, tr = counts["total"], counts["trainable"]
        pct = 100 * tr / t if t else 0
        marker = "  <-- updated" if tr > 0 else ""
        print(f"  {group:<{col-2}} {t:>14,} {tr:>12,} {pct:>6.1f}%{marker}")
    print("-" * (col + 36))
    pct = 100 * trainable_params / total_params if total_params else 0
    print(f"  {'TOTAL':<{col-2}} {total_params:>14,} {trainable_params:>12,} {pct:>6.1f}%")
    print()
    print(f"Trainable: {trainable_params:,} / {total_params:,}  ({pct:.4f}% of model)")

param_summary(model)


Module                                       Total params    Trainable       %
------------------------------------------------------------------------------
  backbone                                  1,235,814,400            0    0.0%
  slot_bank                                     2,097,664    2,097,664  100.0%  <-- updated
------------------------------------------------------------------------------
  TOTAL                                     1,237,912,064    2,097,664    0.2%

Trainable: 2,097,664 / 1,237,912,064  (0.1695% of model)
